# Simulate the 2026 World Cup

This notebook simulates the 2026 World Cup using a fitted match-outcome model from the model-comparison notebook.

Main assumptions:

- Group matches use the model's full 90-minute home/draw/away probabilities.
- Group ranking is deliberately approximate: points, then the pre-tournament FIFA rank. The model does not predict scorelines, so official goal-difference and goals-scored tiebreakers cannot be reproduced.
- The top two teams in each group and the eight best third-placed teams qualify.
- FIFA's official 495-way third-place allocation table determines the Round-of-32 bracket.
- Knockout draws are removed proportionally: each team's win probability is divided by the sum of the two win probabilities.
- Elo ratings and recent form are frozen at their pre-tournament values. Simulated results do not update either input.
- A host is treated as the home team when facing a non-host; all other matches are neutral.
- The biggest upset is the win with the largest positive gap between the winner's and loser's FIFA ranks.

The notebook precomputes probabilities for every possible ordered matchup once. Simulations then use dictionary lookups, which makes large runs much faster.

## 1. Settings

The default run uses the selected logistic model, 10,000 simulations, and a fixed random seed. Change `MODEL_NAME` to `"xgboost"` after running the model-comparison notebook if you want to compare models. Set `SAVE_MATCH_LEVEL_RESULTS = True` only when a full match-level audit trail is needed; at 10,000 runs this CSV exceeds 100 MB. Compact summary tables are saved by default.

In [ ]:
from collections import Counter, defaultdict
from itertools import combinations, permutations
from pathlib import Path
from time import perf_counter

import joblib
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd


def find_project_root():
    '''Find the project root whether the notebook starts there or in 02 Scripts.'''
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "01 Input").is_dir() and (candidate / "02 Scripts").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the project root")


ROOT = find_project_root()
INPUT = ROOT / "01 Input"
TEMP = ROOT / "03 Temp"
OUTPUT = ROOT / "04 Output"
SIMULATION_OUTPUT = OUTPUT / "monte_carlo"
SIMULATION_OUTPUT.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "logit"  # "logit" or "xgboost"
N_SIMULATIONS = 10_000
RANDOM_SEED = 2026
SAVE_MATCH_LEVEL_RESULTS = False
FORM_CUTOFF = pd.Timestamp("2026-06-10")

# The model-comparison notebook exports the fitted model bundles.
MODEL_PATHS = {
    "logit": OUTPUT / "best_logit_model.joblib",
    "xgboost": OUTPUT / "best_xgboost_model.joblib",
}
if MODEL_NAME not in MODEL_PATHS:
    raise ValueError(f"MODEL_NAME must be one of {list(MODEL_PATHS)}")
MODEL_PATH = MODEL_PATHS[MODEL_NAME]
if not MODEL_PATH.is_file():
    raise FileNotFoundError(
        f"Missing {MODEL_PATH.name}. Run 02_compare_models.ipynb first."
    )
RUN_LABEL = f"{MODEL_NAME}_{N_SIMULATIONS:,}".replace(",", "")

print(f"Project root: {ROOT}")
print(f"Model: {MODEL_NAME} ({MODEL_PATH.name})")
print(f"Simulations: {N_SIMULATIONS:,}")

## 2. Load and validate tournament and model inputs

`world_cup_2026_teams.csv` contains the 12 groups, pre-tournament FIFA ranks, and host flags. `world_cup_2026_third_place_routes.csv` contains all 495 possible official allocations of the eight qualifying third-placed teams.

In [ ]:
teams = pd.read_csv(INPUT / "world_cup_2026_teams.csv")
third_place_routes = pd.read_csv(INPUT / "world_cup_2026_third_place_routes.csv")
current_elo = pd.read_csv(OUTPUT / "current_elo_ratings.csv")
model_bundle = joblib.load(MODEL_PATH)

assert len(teams) == 48
assert teams["team"].is_unique
assert teams["fifa_rank"].is_unique
assert teams.groupby("group").size().eq(4).all()
assert set(teams["group"]) == set("ABCDEFGHIJKL")
assert teams["is_host"].sum() == 3
assert len(third_place_routes) == 495
assert third_place_routes["qualifying_groups"].is_unique

missing_elo = sorted(set(teams["team"]) - set(current_elo["team"]))
if missing_elo:
    raise ValueError(f"Tournament teams missing from the Elo table: {missing_elo}")

expected_classes = ["H", "D", "A"]
if list(model_bundle["classes"]) != expected_classes:
    raise ValueError("The model bundle does not use the expected H/D/A class order")

print(f"Loaded {len(teams)} teams and {len(third_place_routes)} third-place routes")
print(
    f"Model bundle: {model_bundle['model_name']} / "
    f"{model_bundle['specification']} / {model_bundle['model_params']}"
)

## 3. Freeze each team's pre-tournament Elo and recent form

The following reconstruction matches the model-comparison notebook's definition of regulation-time form. For each tournament team, it calculates means over the latest five and ten completed matches through 10 June 2026. These values are calculated once and never changed by a simulated match.

In [ ]:
matches = pd.read_parquet(TEMP / "match_level_elo_history.parquet")
matches["date"] = pd.to_datetime(matches["date"])
matches = matches.loc[matches["date"].le(FORM_CUTOFF)].copy()

goalscorers = pd.read_csv(INPUT / "goalscorers.csv", parse_dates=["date"])
goalscorers = goalscorers.loc[goalscorers["date"].le(FORM_CUTOFF)].copy()

match_key = ["date", "home_team", "away_team"]
goalscorers["minute"] = pd.to_numeric(goalscorers["minute"], errors="coerce")
goalscorers["home_goal"] = goalscorers["team"].eq(goalscorers["home_team"]).astype(int)
goalscorers["away_goal"] = goalscorers["team"].eq(goalscorers["away_team"]).astype(int)
goalscorers["home_goal_90"] = (
    goalscorers["team"].eq(goalscorers["home_team"])
    & goalscorers["minute"].notna()
    & goalscorers["minute"].le(90)
).astype(int)
goalscorers["away_goal_90"] = (
    goalscorers["team"].eq(goalscorers["away_team"])
    & goalscorers["minute"].notna()
    & goalscorers["minute"].le(90)
).astype(int)
goalscorers["missing_minute"] = goalscorers["minute"].isna().astype(int)

goal_summary = (
    goalscorers.groupby(match_key, as_index=False)
    .agg(
        event_home_goals=("home_goal", "sum"),
        event_away_goals=("away_goal", "sum"),
        event_home_goals_90=("home_goal_90", "sum"),
        event_away_goals_90=("away_goal_90", "sum"),
        missing_goal_minutes=("missing_minute", "sum"),
    )
)

matches = matches.merge(goal_summary, on=match_key, how="left", validate="one_to_one")
event_columns = [
    "event_home_goals", "event_away_goals",
    "event_home_goals_90", "event_away_goals_90", "missing_goal_minutes",
]
matches[event_columns] = matches[event_columns].fillna(0).astype(int)

complete_events = (
    matches["event_home_goals"].eq(matches["home_score"])
    & matches["event_away_goals"].eq(matches["away_score"])
    & matches["missing_goal_minutes"].eq(0)
)
zero_zero = matches["home_score"].eq(0) & matches["away_score"].eq(0)
from_events = complete_events | zero_zero

matches["home_score_90"] = matches["home_score"]
matches["away_score_90"] = matches["away_score"]
matches.loc[from_events, "home_score_90"] = matches.loc[from_events, "event_home_goals_90"]
matches.loc[from_events, "away_score_90"] = matches.loc[from_events, "event_away_goals_90"]

home_rows = matches[[
    "match_id", "date", "home_team", "away_team",
    "home_score_90", "away_score_90", "home_elo_change",
]].rename(columns={
    "home_team": "team", "away_team": "opponent",
    "home_score_90": "goals_for", "away_score_90": "goals_against",
    "home_elo_change": "elo_change",
})

away_rows = matches[[
    "match_id", "date", "away_team", "home_team",
    "away_score_90", "home_score_90", "away_elo_change",
]].rename(columns={
    "away_team": "team", "home_team": "opponent",
    "away_score_90": "goals_for", "home_score_90": "goals_against",
    "away_elo_change": "elo_change",
})

team_matches = pd.concat([home_rows, away_rows], ignore_index=True)
team_matches = team_matches.sort_values(["team", "date", "match_id"])
team_matches["goal_diff"] = team_matches["goals_for"] - team_matches["goals_against"]
team_matches["points"] = np.select(
    [team_matches["goal_diff"].gt(0), team_matches["goal_diff"].eq(0)],
    [3, 1],
    default=0,
)

form_rows = []
for team, history in team_matches.groupby("team", sort=False):
    row = {"team": team}
    for window in [5, 10]:
        recent = history.tail(window)
        for column in ["goals_for", "goals_against", "goal_diff", "points"]:
            row[f"{column}_{window}"] = recent[column].mean()
        row[f"elo_change_{window}"] = recent["elo_change"].sum()
    form_rows.append(row)

current_form = pd.DataFrame(form_rows)
team_state = (
    teams[["team", "group", "fifa_rank", "is_host"]]
    .merge(current_elo[["team", "elo_rating"]], on="team", validate="one_to_one")
    .merge(current_form, on="team", validate="one_to_one")
)

if team_state.isna().any().any():
    raise ValueError("At least one tournament team has missing Elo or form inputs")

team_state.to_csv(
    SIMULATION_OUTPUT / "world_cup_2026_frozen_team_inputs.csv",
    index=False,
)
display(team_state.sort_values("fifa_rank").head(10))

## 4. Precompute every possible matchup

There are only 48 × 47 = 2,256 ordered pairings. The model pipeline evaluates those once. During simulation, a matchup retrieves three stored probabilities rather than re-running preprocessing and prediction.

The context values are fixed at `is_world_cup = 1`, `is_friendly = 0`, and `k_factor = 60`, matching the World Cup context used by the trained model. These fields describe the match; they do not update team strength.

In [ ]:
state = team_state.set_index("team")
hosts = set(team_state.loc[team_state["is_host"].eq(1), "team"])


def matchup_feature_row(home_team, away_team):
    '''Create one model row using frozen pre-tournament team inputs.'''
    home = state.loc[home_team]
    away = state.loc[away_team]
    home_elo = home["elo_rating"]
    away_elo = away["elo_rating"]

    row = {
        "home_team": home_team,
        "away_team": away_team,
        "home_elo_pre": home_elo,
        "away_elo_pre": away_elo,
        "elo_diff_pre": home_elo - away_elo,
        "abs_elo_diff_pre": abs(home_elo - away_elo),
        "mean_elo_pre": (home_elo + away_elo) / 2,
        "k_factor": 60,
        "neutral": not (home_team in hosts and away_team not in hosts),
        "is_friendly": 0,
        "is_world_cup": 1,
    }

    for side, values in [("home", home), ("away", away)]:
        for window in [5, 10]:
            for feature in [
                "goals_for", "goals_against", "goal_diff", "points", "elo_change"
            ]:
                row[f"{side}_{feature}_{window}"] = values[f"{feature}_{window}"]
    return row


precompute_start = perf_counter()
feature_rows = [
    matchup_feature_row(home_team, away_team)
    for home_team, away_team in permutations(teams["team"], 2)
]
matchup_features = pd.DataFrame(feature_rows)

missing_features = sorted(
    set(model_bundle["feature_columns"]) - set(matchup_features.columns)
)
if missing_features:
    raise ValueError(f"Model features missing from matchup data: {missing_features}")

x_prepared = model_bundle["preprocessor"].transform(
    matchup_features[model_bundle["feature_columns"]]
)
probabilities = model_bundle["model"].predict_proba(x_prepared)

if not np.array_equal(model_bundle["model"].classes_, np.array([0, 1, 2])):
    raise ValueError("Model probability columns are not ordered H/D/A")
if not np.allclose(probabilities.sum(axis=1), 1):
    raise ValueError("Some predicted probabilities do not sum to one")

matchup_probabilities = matchup_features[["home_team", "away_team", "neutral"]].copy()
matchup_probabilities[["p_home_win", "p_draw", "p_away_win"]] = probabilities
probability_lookup = {
    (row.home_team, row.away_team): (row.p_home_win, row.p_draw, row.p_away_win)
    for row in matchup_probabilities.itertuples(index=False)
}

matchup_probabilities.to_csv(
    SIMULATION_OUTPUT / f"world_cup_2026_matchup_probabilities_{MODEL_NAME}.csv",
    index=False,
)

precompute_seconds = perf_counter() - precompute_start
print(
    f"Precomputed {len(matchup_probabilities):,} matchup rows "
    f"in {precompute_seconds:.2f} seconds"
)

## 5. Tournament simulation functions

The functions below separate group matches, knockout matches, bracket routing, and result recording. Group-stage draws receive one point each. When points are equal, the better (lower) FIFA rank is used as the approximation to the official tiebreak sequence.

In [ ]:
rank_lookup = teams.set_index("team")["fifa_rank"].to_dict()
groups = teams.groupby("group", sort=True)["team"].apply(list).to_dict()
route_lookup = third_place_routes.set_index("qualifying_groups").to_dict("index")


def probabilities_for(team_a, team_b):
    '''Return probabilities relative to team_a/team_b, orienting a sole host at home.'''
    swap = team_b in hosts and team_a not in hosts
    home_team, away_team = (team_b, team_a) if swap else (team_a, team_b)
    p_home, p_draw, p_away = probability_lookup[(home_team, away_team)]
    if swap:
        return p_away, p_draw, p_home
    return p_home, p_draw, p_away


def upset_gap(winner, loser):
    '''Positive when the lower-ranked team beats the higher-ranked team.'''
    return max(rank_lookup[winner] - rank_lookup[loser], 0)


def group_match(team_a, team_b, rng, simulation, match_id):
    p_a, p_draw, p_b = probabilities_for(team_a, team_b)
    outcome = rng.choice(["A", "D", "B"], p=[p_a, p_draw, p_b])

    points = {team_a: 0, team_b: 0}
    winner = loser = None
    if outcome == "A":
        points[team_a] = 3
        winner, loser = team_a, team_b
    elif outcome == "B":
        points[team_b] = 3
        winner, loser = team_b, team_a
    else:
        points[team_a] = points[team_b] = 1

    gap = upset_gap(winner, loser) if winner else 0
    record = {
        "simulation": simulation,
        "match_id": match_id,
        "stage": "group",
        "team_a": team_a,
        "team_b": team_b,
        "p_team_a_win_90": p_a,
        "p_draw_90": p_draw,
        "p_team_b_win_90": p_b,
        "p_team_a_advances": np.nan,
        "outcome": outcome,
        "winner": winner,
        "loser": loser,
        "upset_rank_gap": gap,
    }
    return points, record


def knockout_match(team_a, team_b, rng, simulation, match_id, stage):
    p_a, p_draw, p_b = probabilities_for(team_a, team_b)
    p_a_advances = p_a / (p_a + p_b)

    if rng.random() < p_a_advances:
        winner, loser, outcome = team_a, team_b, "A"
    else:
        winner, loser, outcome = team_b, team_a, "B"

    record = {
        "simulation": simulation,
        "match_id": match_id,
        "stage": stage,
        "team_a": team_a,
        "team_b": team_b,
        "p_team_a_win_90": p_a,
        "p_draw_90": p_draw,
        "p_team_b_win_90": p_b,
        "p_team_a_advances": p_a_advances,
        "outcome": outcome,
        "winner": winner,
        "loser": loser,
        "upset_rank_gap": upset_gap(winner, loser),
    }
    return winner, loser, record


def play_knockout_round(pairings, match_ids, stage, rng, simulation):
    winners, losers, records = [], [], []
    for (team_a, team_b), match_id in zip(pairings, match_ids):
        winner, loser, record = knockout_match(
            team_a, team_b, rng, simulation, match_id, stage
        )
        winners.append(winner)
        losers.append(loser)
        records.append(record)
    return winners, losers, records


def round_of_32_pairings(group_order, qualifying_thirds):
    '''Build FIFA's Round-of-32 slots for the simulated qualifying groups.'''
    third_by_group = {group: group_order[group][2] for group in qualifying_thirds}
    route_key = "".join(sorted(qualifying_thirds))
    route = route_lookup[route_key]

    def routed_third(group_winner):
        third_group = route[f"third_for_{group_winner}"]
        return third_by_group[third_group]

    return [
        (group_order["A"][1], group_order["B"][1]),                # M73
        (group_order["E"][0], routed_third("1E")),                # M74
        (group_order["F"][0], group_order["C"][1]),              # M75
        (group_order["C"][0], group_order["F"][1]),              # M76
        (group_order["I"][0], routed_third("1I")),                # M77
        (group_order["E"][1], group_order["I"][1]),              # M78
        (group_order["A"][0], routed_third("1A")),                # M79
        (group_order["L"][0], routed_third("1L")),                # M80
        (group_order["D"][0], routed_third("1D")),                # M81
        (group_order["G"][0], routed_third("1G")),                # M82
        (group_order["K"][1], group_order["L"][1]),              # M83
        (group_order["H"][0], group_order["J"][1]),              # M84
        (group_order["B"][0], routed_third("1B")),                # M85
        (group_order["J"][0], group_order["H"][1]),              # M86
        (group_order["K"][0], routed_third("1K")),                # M87
        (group_order["D"][1], group_order["G"][1]),              # M88
    ]


def simulate_tournament(simulation, rng):
    match_records = []
    group_order = {}
    group_points = {}
    group_match_number = 1

    for group, group_teams in groups.items():
        points = Counter({team: 0 for team in group_teams})
        for team_a, team_b in combinations(group_teams, 2):
            match_points, record = group_match(
                team_a, team_b, rng, simulation, f"G{group}{group_match_number:02d}"
            )
            points.update(match_points)
            match_records.append(record)
            group_match_number += 1

        group_order[group] = sorted(
            group_teams,
            key=lambda team: (-points[team], rank_lookup[team]),
        )
        group_points.update(points)

    third_placed = [group_order[group][2] for group in sorted(groups)]
    best_thirds = sorted(
        third_placed,
        key=lambda team: (-group_points[team], rank_lookup[team]),
    )[:8]

    round_32_teams = [group_order[g][i] for g in groups for i in [0, 1]] + best_thirds
    highest_stage = {team: "group_stage" for team in teams["team"]}
    for team in round_32_teams:
        highest_stage[team] = "round_of_32"

    r32_pairings = round_of_32_pairings(group_order, {
        teams.set_index("team").loc[team, "group"] for team in best_thirds
    })
    r32_winners, _, records = play_knockout_round(
        r32_pairings, range(73, 89), "round_of_32", rng, simulation
    )
    match_records.extend(records)
    for team in r32_winners:
        highest_stage[team] = "round_of_16"

    # Official bracket: M89 W74-W77, M90 W73-W75, ..., M96 W85-W87.
    r16_pairings = [
        (r32_winners[1], r32_winners[4]),
        (r32_winners[0], r32_winners[2]),
        (r32_winners[3], r32_winners[5]),
        (r32_winners[6], r32_winners[7]),
        (r32_winners[10], r32_winners[11]),
        (r32_winners[8], r32_winners[9]),
        (r32_winners[13], r32_winners[15]),
        (r32_winners[12], r32_winners[14]),
    ]
    r16_winners, _, records = play_knockout_round(
        r16_pairings, range(89, 97), "round_of_16", rng, simulation
    )
    match_records.extend(records)
    for team in r16_winners:
        highest_stage[team] = "quarterfinal"

    qf_pairings = [
        (r16_winners[0], r16_winners[1]),
        (r16_winners[4], r16_winners[5]),
        (r16_winners[2], r16_winners[3]),
        (r16_winners[6], r16_winners[7]),
    ]
    qf_winners, _, records = play_knockout_round(
        qf_pairings, range(97, 101), "quarterfinal", rng, simulation
    )
    match_records.extend(records)
    for team in qf_winners:
        highest_stage[team] = "semifinal"

    sf_pairings = [(qf_winners[0], qf_winners[1]), (qf_winners[2], qf_winners[3])]
    sf_winners, sf_losers, records = play_knockout_round(
        sf_pairings, range(101, 103), "semifinal", rng, simulation
    )
    match_records.extend(records)
    for team in sf_winners:
        highest_stage[team] = "final"

    # The third-place match is included in the upset audit, but not in title odds.
    _, _, bronze_record = knockout_match(
        sf_losers[0], sf_losers[1], rng, simulation, 103, "third_place"
    )
    match_records.append(bronze_record)

    champion, _, final_record = knockout_match(
        sf_winners[0], sf_winners[1], rng, simulation, 104, "final"
    )
    match_records.append(final_record)
    highest_stage[champion] = "champion"

    upsets = [record for record in match_records if record["upset_rank_gap"] > 0]
    biggest_gap = max((record["upset_rank_gap"] for record in upsets), default=0)
    biggest_upsets = [
        record for record in upsets if record["upset_rank_gap"] == biggest_gap
    ]

    return champion, highest_stage, biggest_gap, biggest_upsets, match_records

## 6. Run the simulations

If several matches tie for the largest FIFA-rank gap within one simulated tournament, that simulation's weight is divided equally between the tied upset winners. Consequently, the upset shares remain shares of simulations rather than raw event counts.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
simulation_start = perf_counter()

champion_counts = Counter()
stage_counts = defaultdict(Counter)
biggest_upset_weights = Counter()
simulation_rows = []
all_match_records = []

for simulation in range(1, N_SIMULATIONS + 1):
    champion, highest_stage, biggest_gap, biggest_upsets, match_records = (
        simulate_tournament(simulation, rng)
    )
    champion_counts[champion] += 1
    for team, stage in highest_stage.items():
        stage_counts[team][stage] += 1

    if biggest_upsets:
        tied_weight = 1 / len(biggest_upsets)
        for upset in biggest_upsets:
            biggest_upset_weights[upset["winner"]] += tied_weight
        upset_winners = " | ".join(sorted({x["winner"] for x in biggest_upsets}))
        upset_losers = " | ".join(sorted({x["loser"] for x in biggest_upsets}))
        upset_stages = " | ".join(sorted({x["stage"] for x in biggest_upsets}))
    else:
        biggest_upset_weights["No upset"] += 1
        upset_winners = upset_losers = upset_stages = ""

    simulation_rows.append({
        "simulation": simulation,
        "champion": champion,
        "biggest_upset_rank_gap": biggest_gap,
        "biggest_upset_winner": upset_winners,
        "biggest_upset_loser": upset_losers,
        "biggest_upset_stage": upset_stages,
    })
    if SAVE_MATCH_LEVEL_RESULTS:
        all_match_records.extend(match_records)

simulation_seconds = perf_counter() - simulation_start
print(f"Simulated {N_SIMULATIONS:,} tournaments in {simulation_seconds:.2f} seconds")
print(f"Average: {simulation_seconds / N_SIMULATIONS:.4f} seconds per tournament")
print(f"Linear estimate for 10,000: {simulation_seconds / N_SIMULATIONS * 10_000 / 60:.1f} minutes")

## 7. Save detailed CSV outputs

The main tables contain tournament win shares, highest-stage shares, biggest-upset shares, a one-row-per-simulation summary, the precomputed matchup probabilities, and—when enabled—a match-level audit trail.

In [ ]:
stage_order = [
    "group_stage", "round_of_32", "round_of_16",
    "quarterfinal", "semifinal", "final", "champion",
]

win_shares = team_state[["team", "fifa_rank", "elo_rating"]].copy()
win_shares["tournament_wins"] = win_shares["team"].map(champion_counts).fillna(0).astype(int)
win_shares["win_share"] = win_shares["tournament_wins"] / N_SIMULATIONS
win_shares = win_shares.sort_values(["win_share", "fifa_rank"], ascending=[False, True])

stage_rows = []
for team in teams["team"]:
    highest_counts = stage_counts[team]
    highest_index = {stage: i for i, stage in enumerate(stage_order)}
    for stage in stage_order:
        reached = sum(
            count for highest, count in highest_counts.items()
            if highest_index[highest] >= highest_index[stage]
        )
        stage_rows.append({
            "team": team,
            "stage": stage,
            "times_reached": reached,
            "share_reached": reached / N_SIMULATIONS,
        })
stage_reach = pd.DataFrame(stage_rows).merge(
    teams[["team", "fifa_rank"]], on="team", validate="many_to_one"
)

biggest_upset_shares = pd.DataFrame({
    "team": list(biggest_upset_weights),
    "weighted_simulations_as_biggest_upset": list(biggest_upset_weights.values()),
})
biggest_upset_shares["share_of_simulations"] = (
    biggest_upset_shares["weighted_simulations_as_biggest_upset"] / N_SIMULATIONS
)
biggest_upset_shares["fifa_rank"] = biggest_upset_shares["team"].map(rank_lookup)
biggest_upset_shares = biggest_upset_shares.sort_values(
    "share_of_simulations", ascending=False
)

simulation_summary = pd.DataFrame(simulation_rows)
upset_gap_distribution = (
    simulation_summary["biggest_upset_rank_gap"]
    .value_counts()
    .sort_index()
    .rename_axis("biggest_upset_rank_gap")
    .reset_index(name="simulations")
)
upset_gap_distribution["share_of_simulations"] = (
    upset_gap_distribution["simulations"] / N_SIMULATIONS
)

output_tables = {
    f"world_cup_2026_win_shares_{RUN_LABEL}.csv": win_shares,
    f"world_cup_2026_stage_reach_shares_{RUN_LABEL}.csv": stage_reach,
    f"world_cup_2026_biggest_upset_team_shares_{RUN_LABEL}.csv": biggest_upset_shares,
    f"world_cup_2026_biggest_upset_gap_distribution_{RUN_LABEL}.csv": upset_gap_distribution,
    f"world_cup_2026_simulation_summary_{RUN_LABEL}.csv": simulation_summary,
}

if SAVE_MATCH_LEVEL_RESULTS:
    output_tables[f"world_cup_2026_match_results_{RUN_LABEL}.csv"] = pd.DataFrame(
        all_match_records
    )

for filename, table in output_tables.items():
    table.to_csv(SIMULATION_OUTPUT / filename, index=False)
    print(f"Saved {filename}: {len(table):,} rows")

display(win_shares.head(15))
display(biggest_upset_shares.head(15))

## 8. Save PNG charts

Both chart axes show simulation shares. A small smoke test will give noisy shares; the included charts come from the 10,000-run analysis.

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")

win_plot = win_shares.sort_values("win_share", ascending=True)
fig, ax = plt.subplots(figsize=(10, 14))
ax.barh(win_plot["team"], win_plot["win_share"], color="#35618f")
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.set_xlabel("Share of simulated tournaments won")
ax.set_ylabel("")
ax.set_title(f"2026 World Cup title probabilities — {MODEL_NAME} ({N_SIMULATIONS:,} simulations)")
fig.tight_layout()
win_chart_path = SIMULATION_OUTPUT / f"world_cup_2026_win_shares_{RUN_LABEL}.png"
fig.savefig(win_chart_path, dpi=300, bbox_inches="tight")
plt.show()

upset_plot = biggest_upset_shares.loc[
    biggest_upset_shares["team"].ne("No upset")
    & biggest_upset_shares["share_of_simulations"].gt(0)
].sort_values("share_of_simulations", ascending=True)

fig, ax = plt.subplots(figsize=(10, max(7, 0.35 * len(upset_plot))))
ax.barh(upset_plot["team"], upset_plot["share_of_simulations"], color="#a84a3a")
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.set_xlabel("Share of simulations in which team produced the biggest upset")
ax.set_ylabel("")
ax.set_title(f"2026 World Cup biggest-upset shares — {MODEL_NAME} ({N_SIMULATIONS:,} simulations)")
fig.tight_layout()
upset_chart_path = SIMULATION_OUTPUT / f"world_cup_2026_biggest_upset_shares_{RUN_LABEL}.png"
fig.savefig(upset_chart_path, dpi=300, bbox_inches="tight")
plt.show()

print(win_chart_path)
print(upset_chart_path)

## Interpretation cautions

- These are model-based simulation shares, not certainties.
- The group tiebreak approximation can change some qualifiers relative to an official scoreline-based simulation.
- Knockout renormalisation treats the two 90-minute win probabilities as proportional advancement probabilities. It does not explicitly model extra time or penalties.
- FIFA rank is used only for tiebreaks and the upset definition. It is not added as a predictor to the fitted model.
- Fixed Elo and form prevent random simulated results from artificially altering later team strength.